# 01 — Data Understanding

The purpose of this notebook is to understand the raw Excel workbook before cleaning, visualizing, or modelling the data.

In this first section, we only import the required tools, locate the dataset, and inspect its worksheet names.

In [2]:
from pathlib import Path

import pandas as pd

# The notebook is inside the notebooks folder, so its parent folder is the project root.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_file = project_root / "feeder_2023_2025.xlsx"

# Confirm that Python can find the dataset before trying to read it.
if not data_file.exists():
    raise FileNotFoundError(f"Dataset not found: {data_file}")

excel_file = pd.ExcelFile(data_file)

print(f"Dataset: {data_file.name}")
print(f"Worksheets: {excel_file.sheet_names}")

Dataset: feeder_2023_2025.xlsx
Worksheets: ['Hourly_Data', 'Metadata']


## Inspect the metadata sheet

Metadata is information that describes the dataset. Reading it first helps us interpret the hourly measurements correctly and avoids making unsupported assumptions.

In [3]:
metadata = pd.read_excel(excel_file, sheet_name="Metadata")
metadata

,Item,Value
0,Source PDF feeder,UNIVERSITY ROAD / 11KV PARAS
1,Source PDF Sub-station Code No,1633
2,Source PDF Meter Sr No,LT005177
3,Source PDF Class of Voltage,11KV
4,Source PDF Code no of Feeder,05505216331LO2206
5,Second feeder,11KV LAXMINAGAR (synthetic)
6,Second feeder identifiers,Synthetic placeholders because they were not p...
7,Time span,2023-01-01 to 2025-12-31
8,Interval,1 hour
9,Rows per feeder,"26,304"


## Load the hourly feeder data

We now load the main worksheet into a pandas DataFrame named `df`. A DataFrame is a table with rows and columns. Displaying the first five rows lets us inspect the structure without printing the entire dataset.

In [4]:
df = pd.read_excel(excel_file, sheet_name="Hourly_Data")

df.head()

,Date,Hour Interval,Name of Sub-station,Sub-station Code No,Feeder Name,Meter Sr No,Class of Voltage,Code no of Feeder,feeder category,feeder type,...,Frequency (Hz),Power Factor,Q1,Diff in MVArh Q1,Q2,Diff in MVArh Q2,Q3,Diff in MVArh Q3,Q4,Diff in MVArh Q4
0,2023-01-01,00:00,UNIVERSITY ROAD,1633,11KV PARAS,LT005177,11KV,05505216331LO2206,URBN,HT,...,50.000421,0.980554,0.205021,0.005021,0.082869,0.002869,2.812910,0.012910,1.056455,0.006455
1,2023-01-01,01:00,UNIVERSITY ROAD,1633,11KV PARAS,LT005177,11KV,05505216331LO2206,URBN,HT,...,50.016537,0.977000,0.210207,0.005186,0.085832,0.002963,2.826246,0.013336,1.063123,0.006668
2,2023-01-01,02:00,UNIVERSITY ROAD,1633,11KV PARAS,LT005177,11KV,05505216331LO2206,URBN,HT,...,50.016717,0.985331,0.213910,0.003703,0.087948,0.002116,2.835768,0.009522,1.067884,0.004761
3,2023-01-01,03:00,UNIVERSITY ROAD,1633,11KV PARAS,LT005177,11KV,05505216331LO2206,URBN,HT,...,49.994828,0.978327,0.218198,0.004289,0.090399,0.002451,2.846796,0.011028,1.073398,0.005514
4,2023-01-01,04:00,UNIVERSITY ROAD,1633,11KV PARAS,LT005177,11KV,05505216331LO2206,URBN,HT,...,50.024650,0.985041,0.221605,0.003406,0.092345,0.001946,2.855555,0.008759,1.077777,0.004379


## Check the dataset dimensions

The `shape` attribute returns two numbers: `(number of rows, number of columns)`. Rows are hourly feeder observations, while columns are the recorded fields or variables.

In [5]:
rows, columns = df.shape

print(f"Number of rows: {rows:,}")
print(f"Number of columns: {columns}")

Number of rows: 52,608
Number of columns: 34


## List the column names

Column names tell us what information is available in the dataset. We print them one per line with a number so that the full list is easy to inspect.

In [ ]:
for number, column_name in enumerate(df.columns, start=1):
    print(f"{number:>2}. {column_name}")

## Inspect column data types and completeness

The `info()` method summarizes every column. `Non-Null Count` shows how many rows contain a value, and `Dtype` shows how pandas interpreted the values. This is an initial completeness check; we will perform a detailed missing-value calculation later.

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 52608 entries, 0 to 52607
Data columns (total 34 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 52608 non-null  datetime64[us]
 1   Hour Interval        52608 non-null  str           
 2   Name of Sub-station  52608 non-null  str           
 3   Sub-station Code No  52608 non-null  str           
 4   Feeder Name          52608 non-null  str           
 5   Meter Sr No          52608 non-null  str           
 6   Class of Voltage     52608 non-null  str           
 7   Code no of Feeder    52608 non-null  str           
 8   feeder category      52608 non-null  str           
 9   feeder type          52608 non-null  str           
 10  feeder meter_no      52608 non-null  str           
 11  Energy Import        52608 non-null  float64       
 12  Diff MWh Import      52608 non-null  float64       
 13  Energy Export        52608 non-null  int64

## Count missing values

A missing value is an empty or unavailable entry. We count missing values column by column before deciding whether any imputation is necessary.

In [8]:
missing_values = df.isna().sum()
missing_values

Date                   0
Hour Interval          0
Name of Sub-station    0
Sub-station Code No    0
Feeder Name            0
Meter Sr No            0
Class of Voltage       0
Code no of Feeder      0
feeder category        0
feeder type            0
feeder meter_no        0
Energy Import          0
Diff MWh Import        0
Energy Export          0
Diff MWh Export        0
MW(I)                  0
MW(E)                  0
MVA                    0
Current Ir (Amp)       0
Current Iy (Amp)       0
Current Ib (Amp)       0
Voltage Vry (kV)       0
Voltage Vyb (kV)       0
Voltage Vbr (kV)       0
Frequency (Hz)         0
Power Factor           0
Q1                     0
Diff in MVArh Q1       0
Q2                     0
Diff in MVArh Q2       0
Q3                     0
Diff in MVArh Q3       0
Q4                     0
Diff in MVArh Q4       0
dtype: int64

## Check for exact duplicate rows

An exact duplicate occurs when all values in one row are identical to all values in another row. We count such rows before making any decision about removal.

In [9]:
exact_duplicate_count = df.duplicated().sum()

print(f"Number of exact duplicate rows: {exact_duplicate_count}")

Number of exact duplicate rows: 0


## Create a complete timestamp

The date and hour are stored separately. Forecasting requires a single chronological field, so we combine `Date` and `Hour Interval` into a new `timestamp` column. The original columns are retained.

In [10]:
date_text = df["Date"].dt.strftime("%Y-%m-%d")
time_text = df["Hour Interval"].astype(str)

df["timestamp"] = pd.to_datetime(
    date_text + " " + time_text,
    format="%Y-%m-%d %H:%M"
)

df[["Date", "Hour Interval", "timestamp"]].head()

,Date,Hour Interval,timestamp
0,2023-01-01,00:00,2023-01-01 00:00:00
1,2023-01-01,01:00,2023-01-01 01:00:00
2,2023-01-01,02:00,2023-01-01 02:00:00
3,2023-01-01,03:00,2023-01-01 03:00:00
4,2023-01-01,04:00,2023-01-01 04:00:00


## Validate the timestamp column

Before using `timestamp`, we verify its data type and count any failed conversions. A failed conversion would appear as a missing timestamp.

In [11]:
invalid_timestamp_count = df["timestamp"].isna().sum()

print(f"Timestamp data type: {df['timestamp'].dtype}")
print(f"Invalid or missing timestamps: {invalid_timestamp_count}")

Timestamp data type: datetime64[us]
Invalid or missing timestamps: 0


## Check duplicate feeder timestamps

For this dataset, a feeder and timestamp together should identify one observation. Multiple rows with the same feeder code and timestamp would create conflicting hourly readings.

In [13]:
key_columns = ["Code no of Feeder", "timestamp"]
duplicate_key_count = df.duplicated(subset=key_columns).sum()

print(f"Duplicate feeder-timestamp records: {duplicate_key_count}")

Duplicate feeder-timestamp records: 0
